In [ ]:
# Utilities for dataset prep + simple model/dataset classes (Databricks / Ray Train pipeline)
from __future__ import annotations

import os
from typing import Iterable, List, Tuple, Optional
import pandas as pd
from sklearn.model_selection import train_test_split

# Torch bits are only needed for the model/dataset/transforms exports
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

# ---------- Spark → Metadata Join & Path Build ----------

def build_image_metadata_spark_df(
    spark,
    feature_table: str,
    file_table: str,
    image_roots: Iterable[str],
    label_col: str = "label",
    desc_col: str = "DESCRIPTION",
    id_col: str = "ASSET_ID",
    out_path_col: str = "img_path",
):
    """
    Returns a Spark DataFrame with columns [img_path, label] by joining feature & file tables
    and constructing absolute paths for each provided image root. Multiple roots are unioned.
    """
    from pyspark.sql.functions import col, concat, lit

    asset_df = spark.table(feature_table)
    file_df = spark.table(file_table)

    def _one_root_df(root: str):
        return (
            asset_df.join(file_df, id_col)
            .filter(col(desc_col).isNotNull())
            .withColumn(
                out_path_col,
                concat(lit(root), col(id_col).cast("string"), lit(".jpg")),
            )
            .select(out_path_col, label_col)
            .dropna()
        )

    roots = list(image_roots)
    if not roots:
        raise ValueError("image_roots must contain at least one base path.")

    df = _one_root_df(roots[0])
    for r in roots[1:]:
        df = df.unionByName(_one_root_df(r))

    return df


# ---------- Local Filtering (exists) + Stratified Split ----------

def _file_exists(path: str) -> bool:
    try:
        return os.path.isfile(path)
    except Exception:
        return False


def drop_missing_files_pandas(
    img_label_df: pd.DataFrame,
    path_col: str = "img_path",
) -> pd.DataFrame:
    """Keep only rows whose file exists on the driver FS."""
    mask = img_label_df[path_col].apply(_file_exists)
    kept = img_label_df[mask].reset_index(drop=True)
    removed = (~mask).sum()
    print(f"Removed {removed} missing images. Remaining: {len(kept)}.")
    return kept


def stratified_split_pandas(
    img_label_df: pd.DataFrame,
    label_col: str = "label",
    path_col: str = "img_path",
    *,
    min_per_class: int = 2,
    top_n_classes: Optional[int] = 2,
    test_size: float = 0.20,
    val_size: float = 0.20,
    random_state: int = 42,
) -> Tuple[List[Tuple[str, int]], List[Tuple[str, int]], List[Tuple[str, int]]]:
    """
    Filters rare classes, optionally keeps top-N classes, then returns
    (train_ds, val_ds, test_ds) as lists of (img_path, label) tuples using stratified splits.
    """
    vc = img_label_df[label_col].value_counts()
    to_keep = vc[vc >= min_per_class].index
    df = img_label_df[img_label_df[label_col].isin(to_keep)]

    if top_n_classes is not None:
        top_classes = df[label_col].value_counts().head(top_n_classes).index
        df = df[df[label_col].isin(top_classes)]

    print("Final label distribution:")
    print(df[label_col].value_counts())

    # indices for stratified split
    idx = df.index
    train_ids, test_ids = train_test_split(
        idx,
        test_size=test_size,
        stratify=df[label_col],
        random_state=random_state,
    )
    train_ids, val_ids = train_test_split(
        train_ids,
        test_size=val_size,
        stratify=df.loc[train_ids, label_col],
        random_state=random_state,
    )

    def _make_list(_ids):
        sub = df.loc[_ids]
        # ensure int labels
        return list(zip(sub[path_col].tolist(), sub[label_col].astype(int).tolist()))

    train_ds = _make_list(train_ids)
    val_ds = _make_list(val_ids)
    test_ds = _make_list(test_ids)

    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
    return train_ds, val_ds, test_ds


# ---------- One-call convenience: Spark → Pandas → Filter → Split ----------

def prepare_image_dataset(
    spark,
    catalog: str,
    schema: str,
    *,
    feature_table_name: str = "asset_features",
    file_table_name: str = "asset_file_features",
    image_roots: Optional[Iterable[str]] = None,
    min_per_class: int = 2,
    top_n_classes: Optional[int] = 2,
    test_size: float = 0.20,
    val_size: float = 0.20,
    random_state: int = 42,
    path_col: str = "img_path",
    label_col: str = "label",
) -> Tuple[List[Tuple[str, int]], List[Tuple[str, int]], List[Tuple[str, int]], pd.DataFrame]:
    """
    Full pipeline:
    1) Join UC tables and build absolute paths (Spark)
    2) Collect to Pandas
    3) Drop non-existent files
    4) Filter/stratify and return (train_ds, val_ds, test_ds, img_label_df_filtered)
    """
    if image_roots is None:
        image_roots = [
            "/Volumes/shutterstock_dataset_people_detection/sample_datasets/set3_image_people_detection_files/medium/",
            "/Volumes/shutterstock_free_200/sample_datasets/set2_image_object_detection_files/medium/",
        ]

    feature_table = f"{catalog}.{schema}.{feature_table_name}"
    file_table = f"{catalog}.{schema}.{file_table_name}"

    # 1) Spark ETL
    spark_df = build_image_metadata_spark_df(
        spark,
        feature_table=feature_table,
        file_table=file_table,
        image_roots=image_roots,
        label_col=label_col,
        out_path_col=path_col,
    )

    # 2) to Pandas for local checks / sklearn ops
    img_label_df = spark_df.toPandas()

    print("Label distribution (pre-filter):")
    print(img_label_df[label_col].value_counts())

    # 3) drop missing files
    img_label_df = drop_missing_files_pandas(img_label_df, path_col=path_col)

    # 4) stratified split
    train_ds, val_ds, test_ds = stratified_split_pandas(
        img_label_df,
        label_col=label_col,
        path_col=path_col,
        min_per_class=min_per_class,
        top_n_classes=top_n_classes,
        test_size=test_size,
        val_size=val_size,
        random_state=random_state,
    )

    return train_ds, val_ds, test_ds, img_label_df


# ---------- Model / Dataset / Transforms ----------

class SimpleCNN(nn.Module):
    """Minimal CNN for 64x64 RGB → num_classes."""
    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 16 * 16, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(self.conv(x))


class ImageDataset(Dataset):
    """PyTorch dataset that reads images from disk using paths + labels."""
    def __init__(self, img_paths: List[str], labels: List[int], tfms=None):
        self.img_paths = img_paths
        self.labels = labels
        self.tfms = tfms

    def __len__(self) -> int:
        return len(self.img_paths)

    def __getitem__(self, idx: int):
        image = Image.open(self.img_paths[idx]).convert("RGB")
        if self.tfms:
            image = self.tfms(image)
        label = self.labels[idx]
        return image, label


def get_transforms(img_size: Tuple[int, int] = (64, 64)):
    """Returns (train_tfms, test_tfms)."""
    train_tfms = transforms.Compose([
        transforms.Resize(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    test_tfms = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
    ])
    return train_tfms, test_tfms

In [ ]:
train_ds, val_ds, test_ds, img_label_df = prepare_image_dataset(
    spark,
    catalog=SA.catalog_name,
    schema=SA.schema_name,
    # Optional overrides:
    # image_roots=[...], min_per_class=2, top_n_classes=2, test_size=0.2, val_size=0.2
)

# (Optional) grab transforms & model for later cells
train_tfms, test_tfms = get_transforms((64, 64))
model = SimpleCNN(num_classes=2)